# Notebook 7 (Fixed): Final Predictions and Submission

This fixed notebook focuses on deployment only (no retraining):
- Loads model bundles from Notebook 5.
- Reads AMRFinder TSV files from `amr_results/` (or nested `amr_results/amr_results/`).
- Builds prediction vectors safely for each antibiotic.
- Exports:
  - `final_predictions_long.csv` (one row per genome+antibiotic)
  - `submission.csv` (one row per genome with antibiotic-wise predictions)


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

TARGET_ANTIBIOTICS = ['gentamicin', 'ciprofloxacin', 'meropenem']


In [8]:
def discover_amr_tsv_files(base_dir=Path('../amr_results')):
    base_dir = Path(base_dir)
    files = sorted(base_dir.glob('*.tsv')) if base_dir.exists() else []
    if not files:
        nested = base_dir / 'amr_results'
        if nested.exists():
            files = sorted(nested.glob('*.tsv'))
    return files


def parse_genome_id_from_tsv(tsv_path: Path):
    stem = tsv_path.stem.replace('_amr', '')
    try:
        return float(stem)
    except ValueError:
        return np.nan


def extract_genes_from_tsv(tsv_path: Path):
    df = pd.read_csv(tsv_path, sep='\t')
    for col in ['Gene symbol', 'NAME', 'Gene', 'gene']:
        if col in df.columns:
            vals = df[col].dropna().astype(str).str.strip()
            vals = vals[vals != '']
            return set(vals.unique())
    return set()


def load_model_bundles(antibiotics=TARGET_ANTIBIOTICS):
    bundles = {}
    for antibiotic in antibiotics:
        p = Path(f'../models/{antibiotic}_model_bundle.pkl')
        if not p.exists():
            raise FileNotFoundError(f'Missing model bundle: {p}')
        bundle = joblib.load(p)
        if 'model' not in bundle or 'feature_cols' not in bundle:
            raise ValueError(f'Invalid bundle format in: {p}')
        bundles[antibiotic] = bundle
    return bundles


def class_label_mapping(bundle, classes):
    enc = bundle.get('label_encoder', None)

    if isinstance(enc, dict):
        return {c: enc.get(c, enc.get(str(c), str(c))) for c in classes}

    if enc is not None and hasattr(enc, 'inverse_transform'):
        try:
            labels = enc.inverse_transform(classes.astype(int))
            return {c: l for c, l in zip(classes, labels)}
        except Exception:
            pass

    # Common binary fallback used across this project
    class_set = set(classes.tolist())
    if class_set == {0, 1} or class_set == {0.0, 1.0}:
        return {0: 'Susceptible', 1: 'Resistant'}

    return {c: str(c) for c in classes}


def resistance_probability(prob_map):
    # Prefer explicit 'Resistant' class label when available
    for k, v in prob_map.items():
        if str(k).strip().lower() == 'resistant':
            return float(v)

    # Fallback: probability of class 1 if labels are numeric-like
    for k, v in prob_map.items():
        if str(k).strip() in {'1', '1.0'}:
            return float(v)

    # Last fallback: max probability
    return float(max(prob_map.values())) if prob_map else np.nan


def predict_for_genes(genes_present, bundle):
    model = bundle['model']
    feature_cols = bundle['feature_cols']

    x = pd.DataFrame([{
        col: int(col.replace('gene_', '') in genes_present)
        for col in feature_cols
    }])

    proba = model.predict_proba(x)[0]
    classes = np.array(model.classes_)
    pred_class = model.predict(x)[0]

    label_map = class_label_mapping(bundle, classes)
    prob_map = {label_map[c]: float(p) for c, p in zip(classes, proba)}
    pred_label = label_map.get(pred_class, str(pred_class))

    return {
        'prediction': pred_label,
        'confidence': float(np.max(proba)),
        'resistance_probability': resistance_probability(prob_map),
        'probabilities': prob_map,
    }


In [9]:
bundles = load_model_bundles(TARGET_ANTIBIOTICS)
tsv_files = discover_amr_tsv_files()

if not tsv_files:
    raise FileNotFoundError('No TSV files found under ../amr_results/ or ../amr_results/amr_results/')

print('TSV files found:', len(tsv_files))

records = []
for tsv_path in tsv_files:
    try:
        genes = extract_genes_from_tsv(tsv_path)
    except Exception:
        genes = set()

    gid = parse_genome_id_from_tsv(tsv_path)

    for antibiotic, bundle in bundles.items():
        pred = predict_for_genes(genes, bundle)
        records.append({
            'Genome ID': gid,
            'tsv_file': tsv_path.name,
            'antibiotic': antibiotic,
            'prediction': pred['prediction'],
            'confidence': pred['confidence'],
            'resistance_probability': pred['resistance_probability'],
            'probabilities': pred['probabilities'],
        })

pred_long = pd.DataFrame(records)
pred_long = pred_long.sort_values(['Genome ID', 'antibiotic']).reset_index(drop=True)
pred_long.to_csv('../data/processed/final_predictions_long.csv', index=False)

print('Saved ../data/processed/final_predictions_long.csv with rows:', len(pred_long))
pred_long.head()


TSV files found: 274
Saved ../data/processed/final_predictions_long.csv with rows: 822


,Genome ID,tsv_file,antibiotic,prediction,confidence,resistance_probability,probabilities
0,562.100058,562.100058.tsv,ciprofloxacin,Susceptible,0.955671,0.044329,"{'Susceptible': 0.9556710560965604, 'Resistant..."
1,562.100058,562.100058.tsv,gentamicin,Resistant,0.966151,0.966151,"{'Susceptible': 0.033849471230730566, 'Resista..."
2,562.100058,562.100058.tsv,meropenem,Susceptible,0.970521,0.029479,"{'Susceptible': 0.9705211207022215, 'Resistant..."
3,562.100084,562.100084.tsv,ciprofloxacin,Resistant,0.979762,0.979762,"{'Susceptible': 0.020237538111720466, 'Resista..."
4,562.100084,562.100084.tsv,gentamicin,Resistant,0.952495,0.952495,"{'Susceptible': 0.04750495767822982, 'Resistan..."


In [10]:
# Build a compact submission-style table: one row per genome
pred_label_wide = pred_long.pivot(index='Genome ID', columns='antibiotic', values='prediction')
pred_prob_wide = pred_long.pivot(index='Genome ID', columns='antibiotic', values='resistance_probability')

pred_label_wide = pred_label_wide.add_suffix('_prediction')
pred_prob_wide = pred_prob_wide.add_suffix('_resistance_probability')

submission = pd.concat([pred_label_wide, pred_prob_wide], axis=1).reset_index()
submission = submission.sort_values('Genome ID').reset_index(drop=True)
submission.to_csv('../data/processed/submission.csv', index=False)

print('Saved ../data/processed/submission.csv with rows:', len(submission))
submission.head()


Saved ../data/processed/submission.csv with rows: 274


antibiotic,Genome ID,ciprofloxacin_prediction,gentamicin_prediction,meropenem_prediction,ciprofloxacin_resistance_probability,gentamicin_resistance_probability,meropenem_resistance_probability
0,562.100058,Susceptible,Resistant,Susceptible,0.044329,0.966151,0.029479
1,562.100084,Resistant,Resistant,Susceptible,0.979762,0.952495,0.036982
2,562.100106,Susceptible,Susceptible,Susceptible,0.011333,0.007723,0.031912
3,562.100134,Susceptible,Susceptible,Susceptible,0.026426,0.020294,0.047602
4,562.100135,Susceptible,Susceptible,Susceptible,0.020227,0.031528,0.008089


In [11]:
print('Files generated:')
for p in ['../data/processed/final_predictions_long.csv', '../data/processed/submission.csv']:
    fp = Path(p)
    print(f'- {p}:', fp.exists(), '| size bytes:', fp.stat().st_size if fp.exists() else 'NA')

print('Prediction counts by antibiotic/class:')
print(pred_long.groupby(['antibiotic', 'prediction']).size())


Files generated:
- ../data/processed/final_predictions_long.csv: True | size bytes: 130049
- ../data/processed/submission.csv: True | size bytes: 28642
Prediction counts by antibiotic/class:
antibiotic     prediction 
ciprofloxacin  Resistant      120
               Susceptible    154
gentamicin     Resistant       69
               Susceptible    205
meropenem      Resistant       43
               Susceptible    231
dtype: int64
